## orthofinder + inputs -> cafe5

#### CAFE5 pipeline — 1 -> Input preparation & format verification

**Goal of CAFE5:** model gene gain/loss as a birth–death process on a dated tree and detect gene families that
expanded or contracted significantly faster than the genome-wide background — especially on the **marine *Hydrophis*** radiation.

### Pipeline
```
A. OrthoFinder GeneCount.tsv  ->  CAFE count matrix 
B. Filter families with >=100 copies in any species  ->  filtered + large sets
C. Dated ultrametric species tree (IQ-TREE2 LSD2 on OrthoFinder concat alignment, root=170 My)
D. Lambda tree (marine Hydrophis vs background) + OG functional annotation
```


In [1]:
import os, subprocess, pandas as pd, numpy as np, re, collections, dendropy, sys
CAFE   = "/hpcfs/users/a1864358/sanders_lab/phylogenomics/cafe"
VPY    = "/hpcfs/users/a1864358/miniconda/envs/v2r_phylo/bin/python"          
IQ     = "/hpcfs/users/a1864358/miniconda/envs/v2r_phylo/bin/iqtree2"           
OF     = "/hpcfs/users/a1864358/sanders_lab/phylogenomics/orthofinder/orthofinder_prot/primary_transcripts_lifted/OrthoFinder/Results_lifted"
os.chdir(CAFE)
def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:]);  print(r.stderr[-1000:] if r.returncode else "")
    return r
print("cwd:", os.getcwd())

cwd: /scratchdata1/users/a1864358/sanders_lab/phylogenomics/cafe


## OrthoFinder gene-count matrix → CAFE format
- OF gives `OG, <species ...>, total`
- CAFE needs `desc, familyid, <species ...>`

-> _drop `Total` & add `Desc` column_

In [2]:
gc = pd.read_csv(f"{OF}/Orthogroups/Orthogroups.GeneCount.tsv", sep="\t")
sp = [c for c in gc.columns if c not in ("Orthogroup","Total")]
assert (gc[sp].sum(axis=1) == gc["Total"]).all(), "Total column mismatch"
print(f"{len(gc)} orthogroups x {len(sp)} species"); gc.head(3)

19563 orthogroups x 12 species


,Orthogroup,caspera,hcure,hcurw,hcy,hmaj,horn,nhel,nscut,ptext,tele,vberus,vkomodo,Total
0,OG0000000,3,199,264,214,408,229,105,6,24,16,41,1,1510
1,OG0000001,30,166,166,169,170,161,59,44,39,104,38,1,1147
2,OG0000002,23,119,118,122,117,102,85,30,25,37,50,0,828


In [3]:
cafe = pd.DataFrame({"Desc":"(null)", "FamilyID":gc["Orthogroup"]})
for s in sp: cafe[s] = gc[s]
cafe.to_csv("inputs/cafe_counts_all.tsv", sep="\t", index=False)
cafe.head(3)

,Desc,FamilyID,caspera,hcure,hcurw,hcy,hmaj,horn,nhel,nscut,ptext,tele,vberus,vkomodo
0,(null),OG0000000,3,199,264,214,408,229,105,6,24,16,41,1
1,(null),OG0000001,30,166,166,169,170,161,59,44,39,104,38,1
2,(null),OG0000002,23,119,118,122,117,102,85,30,25,37,50,0


---

### filtering
Families with **≥100 gene copies in any single species** = high variance -> destabilise λ (_the model's numerical estimation becomes unreliable_)
- CAFE convention sets them aside into a *large* file analysed separately with a fixed λ (Notebook 2)

- 19,559 of 19,563 families (99.98%) went into the main analysis 
- the filter removed exactly 4 families (= V2R/zinc-finger arrays -> _they evolve by tandem duplication / unequal crossover, not the "one gene born or dies at a time" model CAFE assumes_) 
- —> hence forcing them through => wrong answer, not just no answer 
- filter protects the λ estimate for the 19,559 families

In [4]:
maxc = cafe[sp].max(axis=1)
small = cafe[maxc < 100]; large = cafe[maxc >= 100]
small.to_csv("inputs/cafe_counts_filtered.tsv", sep="\t", index=False)
large.to_csv("inputs/cafe_counts_large.tsv",    sep="\t", index=False)
print(f"filtered (main analysis): {len(small)}   |   large (>=100): {len(large)}")
print("present in only 1 species:", int((cafe[sp].gt(0).sum(axis=1)==1).sum()))
print("present in all 12:",        int((cafe[sp].gt(0).sum(axis=1)==12).sum()))
large.head()

filtered (main analysis): 19559   |   large (>=100): 4
present in only 1 species: 100
present in all 12: 12762


,Desc,FamilyID,caspera,hcure,hcurw,hcy,hmaj,horn,nhel,nscut,ptext,tele,vberus,vkomodo
0,(null),OG0000000,3,199,264,214,408,229,105,6,24,16,41,1
1,(null),OG0000001,30,166,166,169,170,161,59,44,39,104,38,1
2,(null),OG0000002,23,119,118,122,117,102,85,30,25,37,50,0
3,(null),OG0000003,134,33,30,25,32,26,33,41,42,123,47,1


In [5]:
large = pd.read_csv("/hpcfs/users/a1864358/sanders_lab/phylogenomics/cafe/inputs/cafe_counts_large.tsv", sep="\t")
fanno = pd.read_csv("/hpcfs/users/a1864358/sanders_lab/phylogenomics/cafe/inputs/og_annotation.tsv", sep="\t")
large.head(10)

for i in large["FamilyID"]:
    print(f"filtered gene families (>=100 copies in any species): \n")
    print(i)
    subprocess.run(f"grep {i} /hpcfs/users/a1864358/sanders_lab/phylogenomics/cafe/inputs/og_annotation.tsv", shell=True, check=True)

filtered gene families (>=100 copies in any species): 

OG0000000


OG0000000	Vomeronasal type-2 receptor (V2R)	curated_v2r	Vmn2r65	Vmn2r26: Vomeronasal type-2 receptor 26
filtered gene families (>=100 copies in any species): 

OG0000001
OG0000001	ZNF850: Zinc finger protein 850	hydrophis_fallback	nucleic acid binding	ZNF850: Zinc finger protein 850
filtered gene families (>=100 copies in any species): 

OG0000002
OG0000002	Zinc finger (KRAB/C2H2)	eggnog	Zinc finger protein	ZNF665: Zinc finger protein 665
filtered gene families (>=100 copies in any species): 

OG0000003
OG0000003	Vomeronasal type-2 receptor (V2R)	curated_v2r	Vmn2r65	Vmn2r26: Vomeronasal type-2 receptor 26


In [6]:
# CAFE5 filtering script — clade_and_size_filter.py
# (Mendes, CAFE tutorial). That script does the size split (>=100 -> "large") AND a clade filter:
    # keep a family only if >=2 species have >=1 gene (ancestral-state reconstruction needs >=2 species).

work = "inputs/cafe_cladesize"; os.makedirs(work, exist_ok=True)
script = os.path.abspath("notebooks/clade_and_size_filter.py")
inp    = os.path.abspath("inputs/cafe_counts_all.tsv")
# the script writes <name> and large_<name> into its cwd -> run from `work` with a bare basename
subprocess.run([sys.executable, script, "-i", inp, "-o", "cafe_filtered.tsv", "-s"], cwd=work, check=True)

cafe_filt  = pd.read_csv(f"{work}/cafe_filtered.tsv", sep="\t")
cafe_large = pd.read_csv(f"{work}/large_cafe_filtered.tsv", sep="\t")
mine_filt  = pd.read_csv("inputs/cafe_counts_filtered.tsv", sep="\t")
mine_large = pd.read_csv("inputs/cafe_counts_large.tsv", sep="\t")
print(f"CAFE script   : filtered {len(cafe_filt):5d}   large {len(cafe_large)}")
print(f"cell [6] here : filtered {len(mine_filt):5d}   large {len(mine_large)}")

dropped = set(mine_filt.FamilyID) - set(cafe_filt.FamilyID)
nsp = (cafe.set_index("FamilyID").loc[list(dropped), sp] > 0).sum(axis=1)

cafe_filt.head(5)

CAFE script   : filtered 19459   large 4
cell [6] here : filtered 19559   large 4


,Desc,FamilyID,caspera,hcure,hcurw,hcy,hmaj,horn,nhel,nscut,ptext,tele,vberus,vkomodo
0,(null),OG0000004,0,67,52,67,74,64,1,0,1,1,0,0
1,(null),OG0000005,9,35,35,42,36,43,29,10,17,20,37,2
2,(null),OG0000006,28,33,41,30,27,20,28,20,29,4,15,35
3,(null),OG0000007,15,37,46,24,32,32,25,7,12,53,11,4
4,(null),OG0000008,0,35,25,32,32,29,3,20,11,63,2,1


---
---

### OG functional annotation (`inputs/og_annotation.tsv`) 

Merge, in priority order, into the annotation:
1. **Curated V2Rs** — OGs containing a `is_v2r=yes` gene from the V2R curation pipeline
   (`.../curation/results/<sp>/v2r_master.tsv`)
2. **eggNOG** normalized label (all-12-species, orthology-based; paralog names collapsed to families,
   e.g. `Vmn2r65`→V2R, `ZNF*`→zinc finger).
3. **Hydrophis product** fallback, only where eggNOG is uninformative (blank / DUF / "nucleic acid binding" / …).
4. else `uncharacterized`.

In [7]:
# OG annotation: V2R = curation (priority) -> eggNOG (uniform, all 12) -> Hydrophis product (fallback)
import collections
HYD = ["hcure","hcurw","hcy","hmaj","horn"]                 # 5 in-house annotated Hydrophis genomes
CURDIR = "/hpcfs/users/a1864358/sanders_lab/asm/files/annotation/eviann_results/hmaj/curation/results"
ogt = pd.read_csv(f"{OF}/Orthogroups/Orthogroups.tsv", sep="\t").fillna("")

# --- (0) Hydrophis raw-product labels (EviAnn best-hit text) -> og_annotation_hydrophis.tsv (fallback + provenance)
_meta = pd.read_csv("/hpcfs/users/a1864358/sanders_lab/phylogenomics/prot_metadata.tsv",
                    sep="\t", usecols=["transcript_id","product"])
_prod = dict(zip(_meta.transcript_id, _meta["product"].fillna("")))
def _clean(p):
    p = re.sub(r"^Similar to ", "", str(p)); p = re.sub(r"\s*\(.*?\)\s*$", "", p); return p.strip()
_hrows = []
for _, r in ogt.iterrows():
    v = collections.Counter()
    for s in HYD:
        for g in [x.strip() for x in r[s].split(",") if x.strip()]:
            p = _prod.get(g, "")
            if p and "function unknown" not in p.lower() and p.lower() != "nan":
                v[_clean(p)] += 1
    _hrows.append((r["Orthogroup"], v.most_common(1)[0][0] if v else "uncharacterized/no Hydrophis annot"))
pd.DataFrame(_hrows, columns=["FamilyID","consensus_product"]).to_csv(
    "inputs/og_annotation_hydrophis.tsv", sep="\t", index=False)
hyd = pd.read_csv("inputs/og_annotation_hydrophis.tsv", sep="\t").set_index("FamilyID")["consensus_product"]

# --- (1) V2R identity sets: curated Hydrophis gold (species-qualified) + outgroup-only eggNOG V2R
curbysp = {sp: set(pd.read_csv(f"{CURDIR}/{sp}/v2r_master.tsv", sep="\t")
                     .query("is_v2r == 'yes'")["transcript_id"].dropna().astype(str)) for sp in HYD}
gold_v2r, has_hyd = set(), {}
for _, r in ogt.iterrows():
    hy = sum(len([x for x in str(r[sp]).split(",") if x.strip()]) for sp in HYD)
    has_hyd[r["Orthogroup"]] = hy > 0
    if any({x.strip() for x in str(r[sp]).split(",") if x.strip()} & curbysp[sp] for sp in HYD):
        gold_v2r.add(r["Orthogroup"])

# --- (2) eggNOG labels + paralog->family normalisation
egg = pd.read_csv("eggnog/og_annotation_eggnog.tsv", sep="\t").set_index("FamilyID")["eggnog_consensus_label"]
def is_v2r_text(l): return isinstance(l, str) and bool(re.search(r"vmn2r|vomeronasal type-2", l, re.I))
egg_v2r = set(egg.index[egg.map(is_v2r_text)])
v2r_clean = gold_v2r | (egg_v2r & {o for o in has_hyd if not has_hyd[o]})   # curated (Hydrophis) + outgroup-only eggNOG
v2r_like  = egg_v2r - v2r_clean                                             # eggNOG-V2R, Hydrophis-present, curation-excluded

def norm(l):
    if not isinstance(l, str) or l.strip() in ("", "-", "unannotated"): return None
    x = l.lower()
    if re.search(r"\bvmn2r|vomeronasal type-2", x): return "Vomeronasal type-2 receptor (V2R)"
    if re.search(r"\bvmn1r|vomeronasal type-1", x): return "Vomeronasal type-1 receptor (V1R)"
    if re.search(r"olfactory receptor|odorant binding|^or[0-9]|olfr", x): return "Olfactory receptor (OR)"
    if re.search(r"zinc finger|krab|kruppel|krueppel|c2h2|zkscan|zscan|scan domain|\bznf|zinc ion binding", x): return "Zinc finger (KRAB/C2H2)"
    if re.search(r"reverse transcriptase|ribonuclease h|rnase h|dde superfamily|endonuclease|transpos|retro|integrase|\bpol\b|tf2|k02a2", x): return "Transposable element / retroelement"
    if re.search(r"immunoglobulin|\big |ig-|\bmhc|histocompat|hla-|\bmr1\b|antigen|v-set", x): return "Immune (Ig / MHC)"
    if re.search(r"\bgimap", x): return "GIMAP GTPase"
    if re.search(r"c-type lectin|carbohydrate-recognition|\bctl\b|carbohydrate binding", x): return "C-type lectin"
    if re.search(r"cytochrome p450|\bcyp[0-9]", x): return "Cytochrome P450"
    return l
UNINF = re.compile(r"nucleic acid binding|^7 transmembrane|belongs to the g-protein|zinc ion binding|"
                   r"unannotated|unknown|duf[0-9]|domain of unknown|uncharacter|^-$|leucine rich|"
                   r"transmembrane protein|protein of unknown|coiled", re.I)
def uninformative(l): return (not isinstance(l, str)) or l.strip() in ("", "-") or bool(UNINF.search(str(l)))

# --- (3) assemble: V2R(curation) > V2R-like > eggNOG(non-V2R) > Hydrophis product(non-V2R-text) > uncharacterized
rows = []
for fam in ogt["Orthogroup"]:
    e = egg.get(fam); h = hyd.get(fam)
    h_ok = isinstance(h, str) and h != "uncharacterized/no Hydrophis annot" and not is_v2r_text(h)
    ne = norm(e)
    if fam in v2r_clean:                                   src, label = "curated/outgroup_v2r", "Vomeronasal type-2 receptor (V2R)"
    elif fam in v2r_like:                                  src, label = "v2r_like_uncurated", "Class C GPCR / V2R-like (uncurated)"
    elif ne is not None and not uninformative(e):          src, label = "eggnog", ne
    elif h_ok:                                             src, label = "hydrophis_fallback", h
    elif ne is not None:                                   src, label = "eggnog_generic", ne
    else:                                                  src, label = "none", "uncharacterized"
    rows.append((fam, label, src, e, h))

annot = pd.DataFrame(rows, columns=["FamilyID","consensus_product","source","eggnog_raw","hydrophis_label"])
annot.to_csv("inputs/og_annotation.tsv", sep="\t", index=False)
print("wrote inputs/og_annotation.tsv (canonical):", len(annot), "OGs")
print(annot.source.value_counts().to_string())
print("clean V2R OGs:", (annot.source == "curated/outgroup_v2r").sum(),
      "| V2R-like(uncurated):", (annot.source == "v2r_like_uncurated").sum())
annot.head(6)

wrote inputs/og_annotation.tsv (canonical): 19563 OGs
source
eggnog                  19204
hydrophis_fallback        137
v2r_like_uncurated         98
curated/outgroup_v2r       88
none                       20
eggnog_generic             16
clean V2R OGs: 88 | V2R-like(uncurated): 98


,FamilyID,consensus_product,source,eggnog_raw,hydrophis_label
0,OG0000000,Vomeronasal type-2 receptor (V2R),curated/outgroup_v2r,Vmn2r65,Vmn2r26: Vomeronasal type-2 receptor 26
1,OG0000001,ZNF850: Zinc finger protein 850,hydrophis_fallback,nucleic acid binding,ZNF850: Zinc finger protein 850
2,OG0000002,Zinc finger (KRAB/C2H2),eggnog,Zinc finger protein,ZNF665: Zinc finger protein 665
3,OG0000003,Vomeronasal type-2 receptor (V2R),curated/outgroup_v2r,Vmn2r65,Vmn2r26: Vomeronasal type-2 receptor 26
4,OG0000004,Transposable element / retroelement,eggnog,Reverse transcriptase (RNA-dependent DNA polym...,Vmn2r26: Vomeronasal type-2 receptor 26
5,OG0000005,Zinc finger (KRAB/C2H2),eggnog,Zinc finger protein,ZNF420: Zinc finger protein 420


---

#### Lambda tree (marine vs background) 
**λ-tree** assigns each branch an integer rate class: `2` = marine *Hydrophis* clade (incl. its stem), `1` = everything else. Used by the multi-λ model (Notebook 2) to test whether sea snakes evolve gene families at a different rate.

In [8]:
print("lambda tree:", open("trees/lambda_tree_marine.txt").read().strip())

lambda tree: (vkomodo:1,(caspera:1,((((((hcure:2,hcurw:2):2,(hmaj:2,horn:2):2):2,hcy:2):2,(nscut:1,ptext:1):1):1,(nhel:1,tele:1):1):1,vberus:1):1):1);


### Dated, ultrametric species tree

- CAFE requires an ultrametric tree (equal root-to-tip distances) - OrthoFinder's ML tree is in substitutions/site 

-> build **time tree** with **IQ-TREE2's LSD2** least-squares dating on OrthoFinder's concatenated species-tree alignment, --> fixing topology + calibrating the **root (_Varanus-snake split_ at 170 My)**.

In [9]:
# Rebuild only if missing (dating already produced trees/species_tree_ultrametric.nwk)
if not os.path.exists("trees/species_tree_ultrametric.nwk"):
    # C1: rename concat alignment headers (IDs -> species)
    idmap={0:"caspera",1:"hcure",2:"hcurw",3:"hcy",4:"hmaj",5:"horn",6:"nhel",7:"nscut",8:"ptext",9:"tele",10:"vberus",11:"vkomodo"}
    with open(f"{OF}/WorkingDirectory/Alignments_ids/SpeciesTreeAlignment.fa") as f, open("trees/SpeciesTreeAlignment.named.fa","w") as o:
        for ln in f: o.write((">"+idmap[int(ln[1:])]+"\n") if ln.startswith(">") else ln)
    open("trees/dates_tips.txt","w").write("\n".join(f"{t} 0" for t in idmap.values())+"\n")
    # C2: LSD2 dating, contemporaneous tips + root=-170  (fixed OF topology)
    sh(f"cd trees && {IQ} -s SpeciesTreeAlignment.named.fa -te species_tree_ML.nwk -m LG+G4 "
       f"--date dates_tips.txt --date-root -170 --dating LSD -pre dated2 -nt 16 -redo")
    # C3: rescale to root=170, enforce min internal branch 0.2 My, rebuild edges -> exact ultrametric
    sh(f'{VPY} - <<"PY"\n'
       'import dendropy\n'
       't=dendropy.Tree.get(path="trees/dated2.timetree.nwk",schema="newick")\n'
       '[setattr(n,"label",None) for n in t.internal_nodes()]\n'
       'rt=max(l.distance_from_root() for l in t.leaf_node_iter()); sf=170.0/rt\n'
       '[setattr(e,"length",e.length*sf) for e in t.preorder_edge_iter() if e.length is not None]\n'
       't.calc_node_ages(ultrametricity_precision=1e-2)\n'
       'for nd in t.postorder_node_iter():\n'
       '    nd.age=0.0 if nd.is_leaf() else max(nd.age,max(c.age for c in nd.child_nodes())+0.2)\n'
       'for nd in t.preorder_node_iter():\n'
       '    if nd.parent_node is not None: nd.edge.length=nd.parent_node.age-nd.age\n'
       't.write(path="trees/species_tree_ultrametric.nwk",schema="newick",suppress_rooting=True,unquoted_underscores=True,suppress_internal_node_labels=True)\n'
       'PY')
print(open("trees/species_tree_ultrametric.nwk").read())

(vkomodo:169.99961046824916,(caspera:72.39933041824679,((((((hcure:1.6763317583777693,hcurw:1.6763317583777693):0.45416654300988935,(hmaj:1.9304983013876584,horn:1.9304983013876584):0.20000000000000018):0.20000000000000018,hcy:2.330498301387659):10.903094283755392,(nscut:12.68902602770514,ptext:12.68902602770514):0.5445665574379106):26.416101531970362,(nhel:20.158911476676746,tele:20.158911476676746):19.490782640436667):7.435915472862405,vberus:47.08560958997582):25.313720828270974):97.60028005000237);



In [10]:
# Draw the dated tree 
t = dendropy.Tree.get(path="trees/species_tree_ultrametric.nwk", schema="newick", rooting="force-rooted")
print(t.as_ascii_plot(plot_metric='length', width=70))
rt=[l.distance_from_root() for l in t.leaf_node_iter()]
print(f"\nroot-to-tip: min={min(rt):.4f} max={max(rt):.4f}  (=root age, My)")

/------------------------------------------------------------- vkomodo
|                                                                     
|                                  /-------------------------- caspera
|                                  |                                  
|                                  |                         /hcure   
+                                  |                         +        
|                                  |                         |hcurw   
|                                  |                         +        
|                                  |                         |hmaj    
|                                  |                     /---+        
\----------------------------------+                     |   |horn    
                                   |                     |   |        
                                   |           /---------+   \hcy     
                                   |           |         |            
      